In [ ]:
import sys
sys.path.append("/home/habjan.e/TNG/Codes/TNG_workshop")
sys.path.append("/home/habjan.e/TNG/TNG_cluster_dynamics")

import numpy as np
import pandas as pd
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset
from matplotlib.colors import LinearSegmentedColormap

#import TNG_DA

TNG_data_path = '/home/habjan.e/TNG/Data/'


### Plotting font

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "text.usetex": False,
})

#mpl.rcParams.update({
 #   "font.family": "serif",
  #  "font.serif": ["TeX Gyre Pagella", "Book Antiqua", "Palatino Linotype", "DejaVu Serif"]
#})

### Shared constants and catalog helpers


In [ ]:
# Hubble parameters are hard-coded rather than pulled from the TNG API so the
# notebook never depends on the TNG server being reachable.
h_bahamas = 0.7
h_tng     = 0.6774

# Local TNG300-1 group-catalog cache (as-downloaded units, e.g. ckpc/h).
_TNG_CACHE = TNG_data_path + 'TNG_data/' + 'TNG300-1'


def _h5_array(path, dataset):
    with h5py.File(path, 'r') as f:
        return f[dataset][...]


# Figure 1: 8-panel RGB image

### Configuration

Eight simulations (five BAHAMAS, three TNG300 resolutions) with one cluster each.


In [ ]:
MODELS = ["SIDM0.1b", "SIDM0.3b", "SIDM1b", "vdSIDMb", "CDMb",
          "TNG300-1", "TNG300-2", "TNG300-3"]

CLUSTER_IDS = {
    "CDMb":     "002",
    "SIDM0.1b": "003",
    "SIDM0.3b": "001",
    "SIDM1b":   "004",
    "vdSIDMb":  "006",
    "TNG300-1":   "3",
    "TNG300-2":   "4",
    "TNG300-3":   "1",
}

SIMS_DICT = {
    "CDMb":     r"BAHAMAS CDM",
    "SIDM0.1b": r"BAHAMAS SIDM 0.1 cm$^2$ g$^{-1}$",
    "SIDM0.3b": r"BAHAMAS SIDM 0.3 cm$^2$ g$^{-1}$",
    "SIDM1b":   r"BAHAMAS SIDM 1.0 cm$^2$ g$^{-1}$",
    "vdSIDMb":  "BAHAMAS Velocity-Dependent SIDM",
    "TNG300-1": "TNG300-1 CDM",
    "TNG300-2": "TNG300-2 CDM",
    "TNG300-3": "TNG300-3 CDM",
}

# Plot box half-size in Mpc
L_MPC = 3.8
NGRID = 512

BAHAMAS_BOXSIZE_MPC = 400.0


### Load ROCKSTAR subhalo catalogs

Used to apply a substructure mass cut so that only particles belonging to subhalos with $\log_{10}(M_\mathrm{grav,bound} / M_\odot) > 11.75$ are drawn as "substructure". Matches the ROCKSTAR detection floor that BAHAMAS's coarser particle mass can resolve, so TNG doesn't get inflated by low-mass subhalos that BAHAMAS cannot find.

In [ ]:
SUBHALO_COLUMNS = [
    "id", "parent_id", "pos_0", "pos_1", "pos_2", "pos_3", "pos_4", "pos_5",
    "num_p", "mass_grav_est", "mgrav_bound", "vrms", "vmax", "rvmax", "rs",
    "kin_to_pot", "Xoff",
]

_ROCKSTAR_BASE = "/projects/mccleary_group/habjan.e/TNG/Data/rockstar_output"
_BAHAMAS_CLUSTER_IDS = [f"{i:03d}" for i in range(1, 101)]
_TNG_CLUSTER_IDS     = [f"{i:01d}" for i in range(0, 100)]

subhalos_df_by_model = {}

for model in MODELS:
    frames = []
    cluster_iter = _TNG_CLUSTER_IDS if model[:3] == 'TNG' else _BAHAMAS_CLUSTER_IDS
    for cid in cluster_iter:
        if model[:3] == 'TNG':
            path = f"{_ROCKSTAR_BASE}/{model}_rockstar_output/rockstar_subhalos_{cid}.list"
        else:
            path = f"{_ROCKSTAR_BASE}/{model}_rockstar_output/bahamas_rockstar_subhalos_{model}_{cid}.list"
        df = pd.read_csv(path, sep=r"\s+", comment="#", names=SUBHALO_COLUMNS, engine="python")
        df["cluster_id"] = cid
        df["dm_model"] = model
        frames.append(df)
    subhalos_df_by_model[model] = pd.concat(frames, ignore_index=True)

df_CDMb      = subhalos_df_by_model["CDMb"]
df_SIDM01b   = subhalos_df_by_model["SIDM0.1b"]
df_SIDM03b   = subhalos_df_by_model["SIDM0.3b"]
df_SIDM1b    = subhalos_df_by_model["SIDM1b"]
df_vdSIDMb   = subhalos_df_by_model["vdSIDMb"]
df_TNG300_1   = subhalos_df_by_model["TNG300-1"]
df_TNG300_2   = subhalos_df_by_model["TNG300-2"]
df_TNG300_3   = subhalos_df_by_model["TNG300-3"]

In [ ]:
# Sanity check: substructure log-masses for a reference TNG cluster
# should contain values above the cut for the cut to do something.
SUB_MIN_LOG_MASS = 11.75

# A handful of ROCKSTAR entries have mgrav_bound == 0; log10 -> -inf, which the
# comparisons below correctly reject. Silence the expected divide-by-zero.
with np.errstate(divide='ignore'):
    _log_mass_check = np.log10(df_TNG300_1[df_TNG300_1['cluster_id'] == CLUSTER_IDS['TNG300-1']]['mgrav_bound'])

print(f"TNG cluster {CLUSTER_IDS['TNG300-1']} log10(mgrav_bound): min={_log_mass_check.min():.2f}, "
      f"max={_log_mass_check.max():.2f}, "
      f"N>{SUB_MIN_LOG_MASS:.2f}={int((_log_mass_check > SUB_MIN_LOG_MASS).sum())}/"
      f"{len(_log_mass_check)}")
assert (_log_mass_check > SUB_MIN_LOG_MASS).any(), \
    f"No TNG substructures above 10^{SUB_MIN_LOG_MASS:.2f} Msun"


def qualifying_halo_ids(model, cl_id, min_log_mass=SUB_MIN_LOG_MASS):
    df = subhalos_df_by_model[model]
    with np.errstate(divide='ignore'):
        log_mass = np.log10(df['mgrav_bound'])
    sel = (df['cluster_id'] == cl_id) & (log_mass > min_log_mass)
    return df.loc[sel, 'id'].to_numpy()


### Cluster $R_{200}$ and substructure radii

ROCKSTAR `pos_0/1/2` are cluster-centric **comoving Mpc/h** (`rockstar/tng_sub_finder.py`
and `rockstar/bahamas_sub_finder.py` both hand positions to ROCKSTAR in cMpc/h), so
$R_{200}$ has to be in cMpc/h as well:

| source | native units | conversion |
| --- | --- | --- |
| TNG `Group_R_Crit200` | ckpc/h | `/ 1e3` |
| BAHAMAS npz `R200` | cMpc/h | none |

Dividing by $h$ anywhere here puts a spurious factor of $1/h = 1.48$ into $r/R_{200}$.
Verified independently: for BAHAMAS the maximum particle radius in each npz is exactly
$5.000\,R_{200}$ with `R200` used as-is.

Computed once here and reused by Figures 3 and 5.


In [ ]:
# ---------------------------------------------------------------------------
# R_200 per cluster, in comoving Mpc/h to match the ROCKSTAR position units.
# ---------------------------------------------------------------------------
_TNG_R200_cMpc_h = {
    sim: _h5_array(TNG_data_path + 'TNG_data/' + sim + '_Group_R_Crit200.hdf5',
                   'Group/Group_R_Crit200') / 1e3
    for sim in ['TNG300-1', 'TNG300-2', 'TNG300-3']
}


def _bahamas_R200_cMpc_h(model, cluster_ids):
    return np.array([
        float(np.load(f'/projects/mccleary_group/habjan.e/TNG/Data/{model}/GrNm_{cid}.npz')['R200'])
        for cid in cluster_ids
    ])


# model -> {cluster_id (str): R_200 [cMpc/h]}
R200_cMpc_h = {}
for model in MODELS:
    if model.startswith('TNG'):
        _arr = _TNG_R200_cMpc_h[model]
        R200_cMpc_h[model] = {cid: float(_arr[int(cid)]) for cid in _TNG_CLUSTER_IDS}
    else:
        _arr = _bahamas_R200_cMpc_h(model, _BAHAMAS_CLUSTER_IDS)
        R200_cMpc_h[model] = dict(zip(_BAHAMAS_CLUSTER_IDS, _arr.astype(float)))

# Attach cluster-centric radii to every catalog. subhalos_df_by_model holds the
# same objects as df_CDMb ... df_TNG300_3, so those pick the columns up too.
for model, _df in subhalos_df_by_model.items():
    _df['r_cMpc_h'] = np.linalg.norm(_df[['pos_0', 'pos_1', 'pos_2']].to_numpy(), axis=1)
    _df['r_R200'] = _df['r_cMpc_h'] / _df['cluster_id'].map(R200_cMpc_h[model]).to_numpy()

# Figure 6 works from the DS+ `substructure_com_r` column instead of ROCKSTAR
# positions, and that column's h convention has not been checked, so it keeps
# the comoving-Mpc R_200 it has always used.
_R200_TNG_cMpc = _TNG_R200_cMpc_h['TNG300-1'] / h_tng

for model in MODELS:
    _r = subhalos_df_by_model[model]['r_R200']
    print(f"{model:10s} r/R200: med={_r.median():.2f}  p99={_r.quantile(0.99):.2f}  max={_r.max():.2f}")


### Data loaders

Positions are returned in Mpc, centered on the cluster's center of potential.

In [ ]:
def load_bahamas(model, cl_id):
    data = np.load(f"/projects/mccleary_group/habjan.e/TNG/Data/{model}/GrNm_{cl_id}.npz")
    CoP = data['CoP']

    def center(pos):
        d = pos - CoP
        return (d + 0.5 * BAHAMAS_BOXSIZE_MPC) % BAHAMAS_BOXSIZE_MPC - 0.5 * BAHAMAS_BOXSIZE_MPC

    dm_pos   = center(data['dm_pos'])
    gas_pos  = center(data['gas_pos'])
    dm_mass  = np.asarray(data['dm_mass'])
    gas_mass = np.asarray(data['gas_mass'])
    dm_ID    = np.asarray(data['dm_ID'])

    members_path = (
        f"/projects/mccleary_group/habjan.e/TNG/Data/rockstar_output/{model}_rockstar_output/"
        f"bahamas_rockstar_subhalo_members_{model}_{cl_id}.list"
    )
    members = pd.read_csv(members_path, sep=r"\s+", names=["halo_id", "particle_id"])
    keep_halos = qualifying_halo_ids(model, cl_id)
    members = members[members['halo_id'].isin(keep_halos)]

    sub_mask = np.isin(dm_ID, np.asarray(members['particle_id']))
    sub_pos  = dm_pos[sub_mask]
    sub_mass = dm_mass[sub_mask]

    return dm_pos, dm_mass, gas_pos, gas_mass, sub_pos, sub_mass

In [ ]:
def load_tng(model, cl_id):
    cid = int(cl_id)

    if model == 'TNG-Cluster':
        sim_int = 'C'
        file_loc = '/scratch/habjan.e/TNG'
    else:
        sim_int = int(model.rsplit("-", 1)[-1])
        file_loc = '/projects/mccleary_group/habjan.e/TNG'

    h = h_tng

    dm_fname = file_loc + f'/Data/TNG_data/5r200_data-{sim_int}/dm_within_5r200_{cid}'
    gas_fname = file_loc + f'/Data/TNG_data/5r200_data-{sim_int}/gas_within_5r200_{cid}'

    with h5py.File(dm_fname+'.hdf5', 'r') as f:

        if model == 'TNG-Cluster':

            dm_coords_raw= f['DarkMatter']['Coordinates'][:]
            dm_masses_raw = f['DarkMatter']['Masses'][:] * 10**10 / h ### solar masses
            dm_ids = f['DarkMatter']['ParticleIDs'][:]

        else: 

            dm_coords_raw = f['PartType1']['Coordinates'][:]
            ### Hard-coded particle DM mass
            dm_part_mass_dict = {'TNG300-1': 4.0 * 10**7, 'TNG300-2': 3.2 * 10**8, 'TNG300-3': 2.5 * 10**9}
            dm_masses_raw = np.zeros(dm_coords_raw.shape[0]) + dm_part_mass_dict[model] / h
            dm_ids = f['PartType1']['ParticleIDs'][:]

    with h5py.File(gas_fname+'.hdf5', 'r') as f:

        gas_coords_raw = f['PartType0']['Coordinates'][:]
        gas_masses_raw = f['PartType0']['Masses'][:]


    box_size_dict = {'TNG300-1': 205000, 'TNG300-2': 205000, 'TNG300-3': 205000, 'TNG-Cluster': 680000}
    dm_pos_kpc  = TNG_DA.coord_cm_corr(cluster_ind=cid, coordinates=dm_coords_raw, boxsize = box_size_dict[model], sim_in=model)  / h
    gas_pos_kpc = TNG_DA.coord_cm_corr(cluster_ind=cid, coordinates=gas_coords_raw, boxsize = box_size_dict[model], sim_in=model) / h
    dm_pos  = dm_pos_kpc  * 1e-3
    gas_pos = gas_pos_kpc * 1e-3

    dm_mass  = dm_masses_raw
    gas_mass = gas_masses_raw * 1e10 / h

    members_path = (
        f'/projects/mccleary_group/habjan.e/TNG/Data/rockstar_output/{model}_rockstar_output/'
        f'rockstar_subhalo_members_{cid}.list'
    )
    members = pd.read_csv(members_path, sep=r"\s+", names=["halo_id", "particle_id"])
    keep_halos = qualifying_halo_ids(model, cl_id)
    members = members[members['halo_id'].isin(keep_halos)]

    sub_mask = np.isin(dm_ids, np.asarray(members['particle_id']))
    sub_pos  = dm_pos[sub_mask]
    sub_mass = dm_mass[sub_mask]

    return dm_pos, dm_mass, gas_pos, gas_mass, sub_pos, sub_mass

### Projection helper

Mass-weighted projection along the z-axis into an `NGRID x NGRID` grid covering the plot extent.

In [ ]:
def project_2d(positions_mpc, weights, L, ngrid):
    in_box = (
        (positions_mpc[:, 0] >= -L) & (positions_mpc[:, 0] < L) &
        (positions_mpc[:, 1] >= -L) & (positions_mpc[:, 1] < L)
    )
    H, _, _ = np.histogram2d(
        positions_mpc[in_box, 0], positions_mpc[in_box, 1],
        bins=ngrid,
        range=[[-L, L], [-L, L]],
        weights=weights[in_box]
    )
    return H.T  # transpose so rows=y, cols=x for imshow

### Additive color compositing

Each component (DM, gas, substructure) is log-normalized to its own per-panel quantile range and mapped to an intensity in `[0, 1]`. The three intensity maps are then composited additively in RGB space, so each component is equally visible regardless of drawing order and overlap regions mix colors naturally.

In [ ]:
DM_COLOR  = np.array((0.0, 0.0, 0.9))
GAS_COLOR = np.array((0.9, 0.0, 0.0))
SUB_COLOR = np.array((0.0, 0.9, 0.0))


def log_intensity(img, low_q=0.6, high_q=0.999):
    vals = img[img > 0]
    lo = np.nanquantile(vals, low_q)
    hi = np.nanquantile(vals, high_q)

    x = np.clip(img, lo, hi)
    x = np.log10(x / lo) / np.log10(hi / lo)
    return np.clip(x, 0.0, 1.0)


def composite_rgb(layers, gamma=0.8):
    h, w = layers[0][0].shape
    rgb = np.zeros((h, w, 3), dtype=float)

    for intensity, color, weight in layers:
        rgb += weight * intensity[..., None] * color[None, None, :]

    # compress bright pixels instead of hard clipping
    rgb = rgb / np.maximum(rgb.max(axis=-1, keepdims=True), 1.0)

    # optional gamma for contrast
    rgb = np.clip(rgb, 0.0, 1.0) ** gamma
    return rgb

### 8-panel figure

In [ ]:
X_LIMITS = (-L_MPC, L_MPC)
Y_LIMITS = (-L_MPC, L_MPC)
extent_mpc = (-L_MPC, L_MPC, -L_MPC, L_MPC)

if len(MODELS) != 8:
    raise ValueError(f"This layout expects 8 models, but received {len(MODELS)}.")

fig = plt.figure(figsize=(15, 15), constrained_layout=False)

# Six underlying columns allow both the 3-panel and centered 2-panel rows.
gs = fig.add_gridspec(
    3, 6,
    wspace=0.01,
    hspace=0.01,
)

# Each tuple is: (row, starting column, ending column).
panel_positions = [
    # Top row: three panels
    (0, 0, 2),
    (0, 2, 4),
    (0, 4, 6),

    # Middle row: two centered panels
    (1, 1, 3),
    (1, 3, 5),

    # Bottom row: three panels
    (2, 0, 2),
    (2, 2, 4),
    (2, 4, 6),
]

axes = []
reference_ax = None

for row, col_start, col_end in panel_positions:
    if reference_ax is None:
        ax = fig.add_subplot(gs[row, col_start:col_end])
        reference_ax = ax
    else:
        ax = fig.add_subplot(
            gs[row, col_start:col_end],
            sharex=reference_ax,
            sharey=reference_ax,
        )

    axes.append(ax)

for sim, ax in zip(MODELS, axes):
    cl_id = CLUSTER_IDS[sim]

    if sim.startswith("TNG"):
        continue 
        dm_pos, dm_m, gas_pos, gas_m, sub_pos, sub_m = load_tng(
            sim, cl_id
        )
        sub_weight = 0.7
    else:
        dm_pos, dm_m, gas_pos, gas_m, sub_pos, sub_m = load_bahamas(
            sim, cl_id
        )
        sub_weight = 0.7

    dm_map = project_2d(dm_pos, dm_m, L_MPC, NGRID)
    gas_map = project_2d(gas_pos, gas_m, L_MPC, NGRID)
    sub_map = project_2d(sub_pos, sub_m, L_MPC, NGRID)

    rgb = composite_rgb([
        (log_intensity(dm_map),  DM_COLOR,  0.9),
        (log_intensity(gas_map), GAS_COLOR, 0.9),
        (log_intensity(sub_map), SUB_COLOR, sub_weight),
    ])

    ax.set_facecolor("black")
    ax.imshow(
        rgb,
        extent=extent_mpc,
        origin="lower",
        interpolation="nearest",
    )

    ax.set_xlim(*X_LIMITS)
    ax.set_ylim(*Y_LIMITS)

    ax.text(
        0.025, 0.925,
        SIMS_DICT[sim],
        fontsize=12,
        fontweight="semibold",
        color="white",
        transform=ax.transAxes,
    )

# Show tick labels only on the outer panels.
for ax in axes:
    ax.label_outer()

fig.supxlabel("X-coordinate [Mpc]", fontsize=16, fontweight="semibold", y=0.07)
fig.supylabel("Y-coordinate [Mpc]", fontsize=16, fontweight="semibold", x=0.075)

fig.savefig("/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/" "dm_gas_substructure_panels.png", bbox_inches="tight", dpi=350)

plt.show()

# Figure 2: Subhalo CDFs

In [ ]:
def binned_mass_function(x, bins, mode="per_host", n_hosts=1, use_log=True):
    """
    x: values to bin (e.g., log10 M if use_log=True, else M)
    bins: shared bin edges for all datasets (in same units as x)
    mode: "pdf", "per_host", or "cumulative"
    n_hosts: number of host halos represented by this sample (>=1)
    use_log: if True, interprets x & bins as log10; densities are per dex

    Returns: x_center, x_err, y, yerr_lo, yerr_hi
    """
    x = np.asarray(x)
    counts, _ = np.histogram(x, bins=bins)
    widths = np.diff(bins)
    centers = 0.5 * (bins[:-1] + bins[1:])
    xerr = 0.5 * widths

    # Gehrels (1986) approx for 68% CL Poisson errors (asymmetric)
    n = counts.astype(float)
    err_lo_counts = n - np.where(n>0, (np.sqrt(n + 0.75) - 1.0)**2, 0.0)
    err_hi_counts = (np.sqrt(n + 0.75) + 1.0)**2 - n

    if mode == "pdf":
        scale = n.sum()
        y = counts / (scale * widths) if scale > 0 else counts*0.0
        ylo = err_lo_counts / (scale * widths) if scale > 0 else counts*0.0
        yhi = err_hi_counts / (scale * widths) if scale > 0 else counts*0.0
    elif mode == "per_host":
        # differential mass function per host per dex (or per linear unit if use_log=False)
        denom = max(n_hosts, 1)
        y   = counts / (denom * widths)
        ylo = err_lo_counts / (denom * widths)
        yhi = err_hi_counts / (denom * widths)
    elif mode == "cumulative":
        # N(>x) per host; place at left edges’ centers for plotting
        c = counts[::-1].cumsum()[::-1]
        y   = c / max(n_hosts, 1)
        # simple symmetric sqrt(N) (or carry Gehrels cumulatives if desired)
        ylo = np.sqrt(c) / max(n_hosts, 1)
        yhi = ylo
        # for cumulative, xerr isn’t very meaningful:
        xerr = np.zeros_like(centers)
    else:
        raise ValueError("mode must be 'pdf', 'per_host', or 'cumulative'")

    # return arrays with only bins that had any data or keep all (here we keep all)
    return centers, xerr, y, ylo, yhi

In [ ]:
vmax_CDMb = np.array(df_CDMb['vmax'])
vmax_SIDM01b  = np.array(df_SIDM01b ['vmax'])
vmax_SIDM03b  = np.array(df_SIDM03b['vmax'])
vmax_SIDM1b  = np.array(df_SIDM1b ['vmax'])
vmax_vdSIDMb = np.array(df_vdSIDMb['vmax'])
vmax_TNG300_1 = np.array(df_TNG300_1['vmax'])
vmax_TNG300_2 = np.array(df_TNG300_2['vmax'])
vmax_TNG300_3 = np.array(df_TNG300_3['vmax'])

vrms_CDMb = np.array(df_CDMb['vrms'])
vrms_SIDM01b  = np.array(df_SIDM01b ['vrms'])
vrms_SIDM03b  = np.array(df_SIDM03b['vrms'])
vrms_SIDM1b  = np.array(df_SIDM1b ['vrms'])
vrms_vdSIDMb = np.array(df_vdSIDMb['vrms'])
vrms_TNG300_1 = np.array(df_TNG300_1['vrms'])
vrms_TNG300_2 = np.array(df_TNG300_2['vrms'])
vrms_TNG300_3 = np.array(df_TNG300_3['vrms'])

with np.errstate(divide='ignore'):
    mass_CDMb = np.log10(np.array(df_CDMb['mgrav_bound']) / h_bahamas)
    mass_SIDM01b  = np.log10(np.array(df_SIDM01b ['mgrav_bound']) / h_bahamas)
    mass_SIDM03b  = np.log10(np.array(df_SIDM03b['mgrav_bound']) / h_bahamas)
    mass_SIDM1b  = np.log10(np.array(df_SIDM1b ['mgrav_bound']) / h_bahamas)
    mass_vdSIDMb = np.log10(np.array(df_vdSIDMb['mgrav_bound']) / h_bahamas)
    mass_TNG300_1 = np.log10(np.array(df_TNG300_1['mgrav_bound']) / h_tng)
    mass_TNG300_2 = np.log10(np.array(df_TNG300_2['mgrav_bound']) / h_tng)
    mass_TNG300_3 = np.log10(np.array(df_TNG300_3['mgrav_bound']) / h_tng)

### Mass, Velocity Dispersion, and Max Circular Velocity distributions

In [ ]:
num_bins = 50
vmax_low, vmax_up = 250, 450
bins = np.linspace(vmax_low, vmax_up, num_bins)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
plt.subplots_adjust(wspace=0.225)

### Left plot: cumulative max circular velocity function

axins1 = zoomed_inset_axes(ax1, 2, loc=3, bbox_to_anchor=(0.025, 0.075, 0.5, 0.5), bbox_transform=ax1.transAxes)
zoom_min, zoom_max = 360, 400
zoom_y_list = []

for data, color, label in [
    (vmax_CDMb[(vmax_CDMb > vmax_low) & (vmax_CDMb < vmax_up)],   'red',   r"BAHAMAS CDM"),
    (vmax_SIDM01b[(vmax_SIDM01b > vmax_low) & (vmax_SIDM01b < vmax_up)],   'green',   r"BAHAMAS SIDM 0.1 cm$^2$ g$^{-1}$"),
    (vmax_SIDM03b[(vmax_SIDM03b > vmax_low) & (vmax_SIDM03b < vmax_up)],   'orange',   r"BAHAMAS SIDM 0.3 cm$^2$ g$^{-1}$"),
    (vmax_SIDM1b[(vmax_SIDM1b > vmax_low) & (vmax_SIDM1b < vmax_up)],   'black',   r"BAHAMAS SIDM 1.0 cm$^2$ g$^{-1}$"),
    (vmax_vdSIDMb[(vmax_vdSIDMb > vmax_low) & (vmax_vdSIDMb < vmax_up)],'blue',  "BAHAMAS Velocity-Dependent SIDM"),
    (vmax_TNG300_1[(vmax_TNG300_1 > vmax_low) & (vmax_TNG300_1 < vmax_up)], 'purple',  'TNG300-1 CDM'),
    (vmax_TNG300_2[(vmax_TNG300_2 > vmax_low) & (vmax_TNG300_2 < vmax_up)], 'deeppink',  'TNG300-2 CDM'),
    (vmax_TNG300_3[(vmax_TNG300_3 > vmax_low) & (vmax_TNG300_3 < vmax_up)], 'olive',  'TNG300-3 CDM'),
]:
    x_mean, x_std, y, y_err, _ = binned_mass_function(data, bins, mode="cumulative", n_hosts=data.shape[0])

    zoom_y_list.append(y[(x_mean > zoom_min) & (x_mean < zoom_max)])

    # points with both horizontal (std in bin) and vertical (Poisson on density) errors
    #ax1.errorbar(x_mean, y, xerr=x_std, yerr=y_err,
     #            fmt='o', ms=3, capsize=2, elinewidth=1,
      #           color=color, label=label, linestyle='none', alpha=0.9)
    ax1.plot(x_mean, y, c=color, linestyle='--', label=label)
    ax1.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

    axins1.plot(x_mean, y, c=color, linestyle='--')
    axins1.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

axins1.set_xlim(zoom_min, zoom_max)
axins1.set_ylim(np.min(zoom_y_list), np.max(zoom_y_list))
axins1.tick_params(axis='y', which='both', labelleft=False)
axins1.set_yscale('log')
mark_inset(ax1, axins1, loc1=2, loc2=4, fc="none", ec="0.5", lw=1)

### Center plot: cumulative velocity dispersion function

vrms_low, vrms_up = 250, 375
bins = np.linspace(vrms_low, vrms_up, num_bins)

axins2 = zoomed_inset_axes(ax2, 2, loc=3, bbox_to_anchor=(0.025, 0.075, 0.5, 0.5), bbox_transform=ax2.transAxes)
zoom_min, zoom_max = 325, 355
zoom_y_list = []

for data, color, label in [
    (vrms_CDMb[(vrms_CDMb > vrms_low) & (vrms_CDMb < vrms_up)],   'red',   r"BAHAMAS CDM"),
    (vrms_SIDM01b[(vrms_SIDM01b > vrms_low) & (vrms_SIDM01b < vrms_up)],   'green',   r"BAHAMAS SIDM 0.1 cm$^2$ g$^{-1}$"),
    (vrms_SIDM03b[(vrms_SIDM03b > vrms_low) & (vrms_SIDM03b < vrms_up)],   'orange',   r"BAHAMAS SIDM 0.3 cm$^2$ g$^{-1}$"),
    (vrms_SIDM1b[(vrms_SIDM1b > vrms_low) & (vrms_SIDM1b < vrms_up)],   'black',   r"BAHAMAS SIDM 1.0 cm$^2$ g$^{-1}$"),
    (vrms_vdSIDMb[(vrms_vdSIDMb > vrms_low) & (vrms_vdSIDMb < vrms_up)],'blue',  "BAHAMAS Velocity-Dependent SIDM"),
    (vrms_TNG300_1[(vrms_TNG300_1 > vrms_low) & (vrms_TNG300_1 < vrms_up)], 'purple',  'TNG300-1 CDM'),
    (vrms_TNG300_2[(vrms_TNG300_2 > vrms_low) & (vrms_TNG300_2 < vrms_up)], 'deeppink',  'TNG300-2 CDM'),
    (vrms_TNG300_3[(vrms_TNG300_3 > vrms_low) & (vrms_TNG300_3 < vrms_up)], 'olive',  'TNG300-3 CDM'),
]:
    x_mean, x_std, y, y_err, _ = binned_mass_function(data, bins, mode="cumulative", n_hosts=data.shape[0])

    zoom_y_list.append(y[(x_mean > zoom_min) & (x_mean < zoom_max)])

    # points with both horizontal (std in bin) and vertical (Poisson on density) errors
    #ax2.errorbar(x_mean, y, xerr=x_std, yerr=y_err,
     #            fmt='o', ms=3, capsize=2, elinewidth=1,
      #           color=color, label=label, linestyle='none', alpha=0.9)
    ax2.plot(x_mean, y, c=color, linestyle='--', label=label)
    ax2.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

    axins2.plot(x_mean, y, c=color, linestyle='--')
    axins2.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

axins2.set_xlim(zoom_min, zoom_max)
axins2.set_ylim(np.min(zoom_y_list), np.max(zoom_y_list))
axins2.tick_params(axis='y', which='both', labelleft=False)
axins2.set_yscale('log')
mark_inset(ax2, axins2, loc1=2, loc2=4, fc="none", ec="0.5", lw=1)

### Right plot: cumulative mass function

mass_low, mass_up = 11.75, 13
bins = np.linspace(mass_low, mass_up, num_bins)

axins3 = zoomed_inset_axes(ax3, 2, loc=3, bbox_to_anchor=(0.025, 0.075, 0.5, 0.5), bbox_transform=ax3.transAxes)
zoom_min, zoom_max = 12, 12.5
zoom_y_list = []

for data, color, label in [
    (mass_CDMb[(mass_CDMb > mass_low) & (mass_CDMb < mass_up)],   'red',   r"BAHAMAS CDM"),
    (mass_SIDM01b[(mass_SIDM01b > mass_low) & (mass_SIDM01b < mass_up)],   'green',   r"BAHAMAS SIDM" + '\n' + r"0.1 cm$^2$ g$^{-1}$"),
    (mass_SIDM03b[(mass_SIDM03b > mass_low) & (mass_SIDM03b < mass_up)],   'orange',   r"BAHAMAS SIDM" + '\n' + r"0.3 cm$^2$ g$^{-1}$"),
    (mass_SIDM1b[(mass_SIDM1b > mass_low) & (mass_SIDM1b < mass_up)],   'black',   r"BAHAMAS SIDM" + '\n' + r"1.0 cm$^2$ g$^{-1}$"),
    (mass_vdSIDMb[(mass_vdSIDMb > mass_low) & (mass_vdSIDMb < mass_up)],'blue',  "BAHAMAS SIDM" + '\n' + r"Velocity-Dependent"),
    (mass_TNG300_1[(mass_TNG300_1 > mass_low) & (mass_TNG300_1 < mass_up)], 'purple',  'TNG300-1 CDM'),
    (mass_TNG300_2[(mass_TNG300_2 > mass_low) & (mass_TNG300_2 < mass_up)], 'deeppink',  'TNG300-2 CDM'),
    (mass_TNG300_3[(mass_TNG300_3 > mass_low) & (mass_TNG300_3 < mass_up)], 'olive',  'TNG300-3 CDM'),
]:
    x_mean, x_std, y, y_err, _ = binned_mass_function(data, bins, mode="cumulative", n_hosts=data.shape[0])

    zoom_y_list.append(y[(x_mean > zoom_min) & (x_mean < zoom_max)])

    # points with both horizontal (std in bin) and vertical (Poisson on density) errors
    #ax3.errorbar(x_mean, y, xerr=x_std, yerr=y_err,
     #            fmt='o', ms=3, capsize=2, elinewidth=1,
      #           color=color, label=label, linestyle='none', alpha=0.9)
    
    ax3.plot(x_mean, y, c=color, linestyle='--', label=label)
    ax3.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

    axins3.plot(x_mean, y, c=color, linestyle='--')
    axins3.fill_between(x_mean, y - y_err, y + y_err, color=color, alpha=0.3)

axins3.set_xlim(zoom_min, zoom_max)
axins3.set_ylim(np.min(zoom_y_list), np.max(zoom_y_list))
axins3.tick_params(axis='y', which='both', labelleft=False)
axins3.set_yscale('log')
mark_inset(ax3, axins3, loc1=2, loc2=4, fc="none", ec="0.5", lw=1)

ax1.set_yscale('log')
ax2.set_yscale('log')
ax3.set_yscale('log')

f_size = 20

ax1.set_ylabel(r'N(> $V_{\rm max}$)', fontweight='semibold', fontsize = f_size)
ax1.set_xlabel(r'Subhalo $V_{\rm max}$ [$km$ $s^{-1}$]', fontweight='semibold', fontsize = f_size)

ax2.set_ylabel(r'N(> $\sigma_v$)', fontweight='semibold', fontsize = f_size)
ax2.set_xlabel(r'Subhalo $\sigma_v$ [$km$ $s^{-1}$]', fontweight='semibold', fontsize = f_size)

ax3.set_ylabel(r'N(> $\log M_{\odot}$)', fontweight='semibold', fontsize = f_size)
ax3.set_xlabel(r'Subhalo Mass [$\log(M_{\odot})$]', fontweight='semibold', fontsize = f_size)

#ax1.legend(loc='lower left')
#ax2.legend(loc='lower left')
ax3.legend(bbox_to_anchor=(1.05, 1.05), fontsize = 15, labelspacing=1.25)

fig.savefig(
    "/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/dm_sub_cdfs.png",
    bbox_inches="tight", dpi=500,
)

# Figure 3: $\sigma_v$ versus $V_{\rm max}$, $\sigma_v$ versus $M_{200}$

In [ ]:
# Subhalo sigma_v against subhalo mass (left) and V_max (right). The greyscale
# 2D histogram is every subhalo of all eight simulations stacked; the eight
# dashed lines are the per-simulation medians with bootstrapped 68% errors.
# Colours and line styles match Figure 2.

N_BOOT_MED  = 1000   # bootstrap resamplings used for the median errors
MIN_PER_BIN = 10     # bins holding fewer subhalos than this are dropped
SEED_MED    = 0

SCALING_SIMS = [
    (mass_CDMb,     vmax_CDMb,     vrms_CDMb,     'red',      r"BAHAMAS CDM"),
    (mass_SIDM01b,  vmax_SIDM01b,  vrms_SIDM01b,  'green',    r"BAHAMAS SIDM" + '\n' + r"0.1 cm$^2$ g$^{-1}$"),
    (mass_SIDM03b,  vmax_SIDM03b,  vrms_SIDM03b,  'orange',   r"BAHAMAS SIDM" + '\n' + r"0.3 cm$^2$ g$^{-1}$"),
    (mass_SIDM1b,   vmax_SIDM1b,   vrms_SIDM1b,   'black',    r"BAHAMAS SIDM" + '\n' + r"1.0 cm$^2$ g$^{-1}$"),
    (mass_vdSIDMb,  vmax_vdSIDMb,  vrms_vdSIDMb,  'blue',     r"BAHAMAS SIDM" + '\n' + r"Velocity-Dependent"),
    (mass_TNG300_1, vmax_TNG300_1, vrms_TNG300_1, 'purple',   'TNG300-1 CDM'),
    (mass_TNG300_2, vmax_TNG300_2, vrms_TNG300_2, 'deeppink', 'TNG300-2 CDM'),
    (mass_TNG300_3, vmax_TNG300_3, vrms_TNG300_3, 'olive',    'TNG300-3 CDM'),
]


def _scaling_mask(mass, vmax, vrms):
    """Shared subhalo selection, so both panels describe an identical sample.

    log10(mgrav_bound) is -inf for the handful of entries with mgrav_bound == 0,
    and the mass floor is the same substructure cut used elsewhere in the
    notebook (BAHAMAS cannot resolve subhalos below it, so leaving it out would
    let TNG's extra low-mass objects pull its medians down).
    """
    return np.isfinite(mass) & (mass > SUB_MIN_LOG_MASS) & (vmax > 0) & (vrms > 0)


def binned_median_boot(x, y, bins, n_boot=N_BOOT_MED, min_count=MIN_PER_BIN, seed=SEED_MED):
    """Median of y per x bin with 16th/84th percentiles of the bootstrapped median.

    Returns centers, median, lo, hi for the bins holding at least min_count points.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    rng = np.random.default_rng(seed)

    centers, med, lo, hi = [], [], [], []
    idx = np.digitize(x, bins) - 1
    for i in range(len(bins) - 1):
        yy = y[idx == i]
        if yy.size < min_count:
            continue
        boot = np.median(rng.choice(yy, size=(n_boot, yy.size), replace=True), axis=1)
        centers.append(0.5 * (bins[i] + bins[i + 1]))
        med.append(np.median(yy))
        lo.append(np.percentile(boot, 16))
        hi.append(np.percentile(boot, 84))

    return np.array(centers), np.array(med), np.array(lo), np.array(hi)


# Bin edges: linear in log10(M) on the left, logarithmic in V_max on the right,
# with sigma_v binned logarithmically in both panels so the histogram cells are
# square on the log axes.
mass_bins = np.linspace(SUB_MIN_LOG_MASS, 14.0, 31)
vmax_bins = np.logspace(np.log10(40), np.log10(750), 31)
vrms_bins = np.logspace(np.log10(60), np.log10(1000), 31)

# Light greyscale so the coloured median lines stay legible on top of it.
hist_cmap = LinearSegmentedColormap.from_list('sub_greys', ['white', '0.45'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
plt.subplots_adjust(wspace=0.225)

_all_mass = np.concatenate([m[_scaling_mask(m, vx, vr)] for m, vx, vr, _, _ in SCALING_SIMS])
_all_vmax = np.concatenate([vx[_scaling_mask(m, vx, vr)] for m, vx, vr, _, _ in SCALING_SIMS])
_all_vrms = np.concatenate([vr[_scaling_mask(m, vx, vr)] for m, vx, vr, _, _ in SCALING_SIMS])

h1 = ax1.hist2d(_all_mass, _all_vrms, bins=[mass_bins, vrms_bins],
                cmap=hist_cmap, norm=mpl.colors.LogNorm(), zorder=0)
h2 = ax2.hist2d(_all_vmax, _all_vrms, bins=[vmax_bins, vrms_bins],
                cmap=hist_cmap, norm=mpl.colors.LogNorm(), zorder=0)

for mass, vmax, vrms, color, label in SCALING_SIMS:
    sel = _scaling_mask(mass, vmax, vrms)

    x, med, lo, hi = binned_median_boot(mass[sel], vrms[sel], mass_bins)
    ax1.plot(x, med, c=color, linestyle='--', lw=1.8, label=label, zorder=3)
    ax1.fill_between(x, lo, hi, color=color, alpha=0.3, lw=0, zorder=2)

    x, med, lo, hi = binned_median_boot(vmax[sel], vrms[sel], vmax_bins)
    ax2.plot(x, med, c=color, linestyle='--', lw=1.8, label=label, zorder=3)
    ax2.fill_between(x, lo, hi, color=color, alpha=0.3, lw=0, zorder=2)

ax1.set_yscale('log')
ax2.set_xscale('log')
ax2.set_yscale('log')

ax1.set_xlim(mass_bins[0], mass_bins[-1])
ax1.set_ylim(vrms_bins[0], vrms_bins[-1])
ax2.set_xlim(vmax_bins[0], vmax_bins[-1])
ax2.set_ylim(vrms_bins[0], vrms_bins[-1])

# Decade-only labels are unreadable over this narrow a dynamic range, so the
# log axes get explicit ticks written as plain numbers.
for _ax in (ax1, ax2):
    _ax.set_yticks([100, 200, 400, 800])
    _ax.yaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
    _ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax2.set_xticks([50, 100, 200, 400])
ax2.xaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
ax2.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())

f_size = 20

ax1.set_xlabel(r'Subhalo Mass [$\log(M_{\odot})$]', fontweight='semibold', fontsize=f_size)
ax1.set_ylabel(r'Subhalo $\sigma_v$ [$km$ $s^{-1}$]', fontweight='semibold', fontsize=f_size)

ax2.set_xlabel(r'Subhalo $V_{max}$ [$km$ $s^{-1}$]', fontweight='semibold', fontsize=f_size)
ax2.set_ylabel(r'Subhalo $\sigma_v$ [$km$ $s^{-1}$]', fontweight='semibold', fontsize=f_size)

cbar = fig.colorbar(h2[3], ax=ax2, pad=0.02)
cbar.set_label(r'N$_{\rm subhalos}$', fontweight='semibold', fontsize=f_size)

ax2.legend(bbox_to_anchor=(1.25, 1.05), fontsize=15, labelspacing=1.25)

fig.savefig(
    "/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/sigma_v_scalings.png",
    bbox_inches="tight", dpi=500,
)

plt.show()


# Figure 4: Radial CDF of ROCKSTAR substructures


In [ ]:
# Radial CDF of ROCKSTAR substructures in three radial selections, left to
# right: every subhalo, the outer sample (r/R_200 > 10^-2), and the inner
# sample (r/R_200 < 10^-2). Each column keeps its own radial grid; the two
# rows share y-axis extents across all three columns.
R_MAX_1, R_MAX_2, R_MAX_3           = 1.5, 1.5, 1e-2
R_R200_MIN_1, R_R200_MIN_2, R_R200_MIN_3 = 0.0, 1e-2, 0.0

SUB_MASS_LO, SUB_MASS_HI = 11, 15
GRID_N       = 16            # radial grid points per panel
GRID_1       = np.linspace(R_R200_MIN_1, R_MAX_1, GRID_N)
GRID_2       = np.linspace(R_R200_MIN_2, R_MAX_2, GRID_N)
GRID_3       = np.linspace(R_R200_MIN_3, R_MAX_3, GRID_N)

REFERENCE    = "TNG300-1"     # curve that the bottom panel is differenced against
N_BOOT       = 500            # 200 is enough for 68%; 500 smooths the band edges
BAND_ALPHA   = 0.3
SHOW_BANDS_TOP    = True
SHOW_BANDS_BOTTOM = True
SEED         = 0

# Column holding the host-halo ID. Set to a string to force it, None to auto-detect,
# or False to deliberately bootstrap over subhalos instead.
#
# This has to be 'cluster_id', not ROCKSTAR's 'parent_id'. ROCKSTAR was run once
# per cluster, so parent_id is only unique *within* a cluster and its values
# repeat across clusters: in TNG300-1 the 100 clusters carry just 49 distinct
# parent_id values, and parent_id 4, 10 and 20 each pool ~190 subhalos drawn
# from several different clusters. Blocking on it therefore resamples ~50
# oversized blocks instead of the 100 independent hosts, which inflates the
# bands. (cluster_id, parent_id) recovers exactly 100 groups, i.e. one per
# cluster, so cluster_id alone is the right block label.
HOST_COL     = 'cluster_id'
_HOST_CANDIDATES = ['host_id', 'hostid', 'host', 'group_id', 'groupid',
                    'GroupNumber', 'halo_id', 'cluster_id', 'fof_id']

RADIAL_SIMS = [
    ("CDMb",     'red',      '--', r"BAHAMAS CDM"),
    ("vdSIDMb",  'blue',     '--', r"BAHAMAS SIDM Velocity-Dependent"),
    ("SIDM0.1b", 'green',    '--', r"BAHAMAS SIDM 0.1 cm$^2$ g$^{-1}$"),
    ("SIDM0.3b", 'orange',   '--', r"BAHAMAS SIDM 0.3 cm$^2$ g$^{-1}$"),
    ("SIDM1b",   'black',    '--', r"BAHAMAS SIDM 1.0 cm$^2$ g$^{-1}$"),
    ("TNG300-1", 'purple',   '--', 'TNG300-1 CDM'),
    ("TNG300-2", 'deeppink', '--', 'TNG300-2 CDM'),
    ("TNG300-3", 'olive',    '--', 'TNG300-3 CDM'),
]

# (r_min, r_max, grid, corner annotation) for each column.
RADIAL_PANELS = [
    (R_R200_MIN_1, R_MAX_1, GRID_1, 'All Subhalos'),
    (R_R200_MIN_2, R_MAX_2, GRID_2, r'$r_{\rm sub}^{\rm rockstar}/R_{200} > 10^{-2}$'),
    (R_R200_MIN_3, R_MAX_3, GRID_3, r'$r_{\rm sub}^{\rm rockstar}/R_{200} < 10^{-2}$'),
]

# ---------------------------------------------------------------------------
# Sample selection
# ---------------------------------------------------------------------------
def _find_host_col(df):
    if HOST_COL is False:
        return None
    if isinstance(HOST_COL, str):
        return HOST_COL if HOST_COL in df.columns else None
    for c in _HOST_CANDIDATES:
        if c in df.columns:
            return c
    return None


def radial_sample(model, mass_lo=SUB_MASS_LO, mass_hi=SUB_MASS_HI,
                  r_min=R_R200_MIN_1, r_max=R_MAX_1):
    """Return (r, host) for the selected substructures. host is None if unavailable."""
    df = subhalos_df_by_model[model]
    with np.errstate(divide='ignore'):
        log_m = np.log10(df['mgrav_bound'].to_numpy())
    r = df['r_R200'].to_numpy()
    keep = (np.isfinite(r) & (r >= r_min) & (r < r_max)
            & (log_m >= mass_lo) & (log_m < mass_hi))

    hcol = _find_host_col(df)
    host = df[hcol].to_numpy()[keep] if hcol is not None else None
    return r[keep], host


def cdf(r, grid):
    """Empirical CDF of r evaluated on grid."""
    return np.searchsorted(np.sort(r), grid, side='right') / r.size

# ---------------------------------------------------------------------------
# Bootstrap
# ---------------------------------------------------------------------------
def bootstrap_cdf(r, host, grid, n_boot=N_BOOT, rng=None):
    """
    Empirical CDF plus bootstrap replicates.

    If `host` is given, whole hosts are resampled with replacement (block
    bootstrap), which respects the fact that subhalos cluster within hosts.
    Otherwise individual subhalos are resampled, which will understate the
    uncertainty by roughly sqrt(mean subhalos per host) in the worst case.

    Returns (cdf, boot) with boot.shape == (n_boot, grid.size).
    """
    rng = np.random.default_rng() if rng is None else rng
    point = cdf(r, grid)

    if host is None:
        n = r.size
        r_sorted_pool = r
        boot = np.empty((n_boot, grid.size))
        for i in range(n_boot):
            boot[i] = cdf(rng.choice(r_sorted_pool, size=n, replace=True), grid)
        return point, boot

    # Host-level block bootstrap, vectorised: precompute each host's contribution
    # to the unnormalised CDF, then resample rows of that matrix.
    uniq, inv = np.unique(host, return_inverse=True)
    n_hosts = uniq.size
    counts_cum = np.zeros((n_hosts, grid.size))   # N(<grid) per host
    n_per_host = np.zeros(n_hosts)
    for j in range(n_hosts):
        rj = np.sort(r[inv == j])
        n_per_host[j] = rj.size
        counts_cum[j] = np.searchsorted(rj, grid, side='right')

    idx = rng.integers(0, n_hosts, size=(n_boot, n_hosts))
    num = counts_cum[idx].sum(axis=1)             # (n_boot, grid.size)
    den = n_per_host[idx].sum(axis=1)[:, None]
    boot = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
    return point, boot


def band(boot, lo=16, hi=84):
    """Percentile envelope of a set of bootstrap replicates."""
    return np.percentile(boot, [lo, hi], axis=0)

# ---------------------------------------------------------------------------
# Build everything, one selection per column
# ---------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
panel_cdfs, panel_boots = [], []

for r_min, r_max, grid, note in RADIAL_PANELS:
    samples, cdfs, boots = {}, {}, {}
    for model, _, _, _ in RADIAL_SIMS:
        r, host = radial_sample(model, r_min=r_min, r_max=r_max)
        samples[model] = (r, host)
        cdfs[model], boots[model] = bootstrap_cdf(r, host, grid, rng=rng)
        mode = f"{np.unique(host).size} hosts" if host is not None else "subhalo-level (!)"
        print(f"[{note}] {model:10s} N={r.size:6d}   bootstrap: {mode}")

    if any(h is None for _, h in samples.values()):
        print("\nWarning: no host-ID column found for at least one model; those bands "
              "are subhalo-level and will be too narrow. Set HOST_COL to fix.")

    panel_cdfs.append(cdfs)
    panel_boots.append(boots)
    print()

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(
    2, 3, figsize=(16.5, 7.5), sharex='col', sharey=False,
    gridspec_kw=dict(height_ratios=[2, 1], hspace=0.05, wspace=0.17),
)

for j, ((r_min, r_max, grid, note), cdfs, boots) in enumerate(
        zip(RADIAL_PANELS, panel_cdfs, panel_boots)):

    ax1, ax2 = axes[0, j], axes[1, j]

    ref      = cdfs[REFERENCE]
    ref_boot = boots[REFERENCE]

    for model, color, ls, label in RADIAL_SIMS:
        c = cdfs[model]
        ax1.plot(grid, c, color=color, ls=ls, lw=1.8, label=label)
        ax2.plot(grid, c - ref, color=color, ls=ls, lw=1.8)

        if SHOW_BANDS_TOP:
            lo, hi = band(boots[model])
            ax1.fill_between(grid, lo, hi, color=color, alpha=BAND_ALPHA, lw=0)

        if SHOW_BANDS_BOTTOM:# and model != REFERENCE:
            # replicate-wise difference: carries model and reference uncertainty
            dlo, dhi = band(boots[model] - ref_boot)
            ax2.fill_between(grid, dlo, dhi, color=color, alpha=BAND_ALPHA, lw=0)

    # Selection label in the bottom-right corner of the CDF panel.
    ax1.text(0.99, 0.02, note, transform=ax1.transAxes,
             ha='right', va='bottom', fontsize=12)

    ax1.set_xlim(r_min, r_max)
    ax2.set_xlim(r_min, r_max)

    # Every panel carries its own y scale: the inner selection's residuals are
    # an order of magnitude larger than the other two, and on a shared scale
    # they flatten the first two Delta-P panels into straight lines. The CDF
    # row is pinned to [0, 1.02] because a CDF spans the same range anyway.
    ax1.set_ylim(0, 1.02)

axes[0, 0].set_ylabel(r'$P\,(<r_{\rm sub}^{\rm rockstar}/R_{200})$', fontsize=20)
axes[1, 0].set_ylabel(r'$\Delta P$', fontsize=20)
fig.supxlabel(r'$r_{\rm sub}^{\rm rockstar}/R_{200}$', fontsize=20, y=0.02)

for a in axes.ravel():
    a.minorticks_on()
    a.tick_params(labelsize=12)

axes[0, 0].legend(fontsize=11, loc='upper left', frameon=False)

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/substructure_radial_cdf.png',
    bbox_inches='tight',
    dpi=500,
    )
plt.show()


# Figure 5: radial offset versus cross section for substructures in $10^{-2} \times R_{200}$ 

In [ ]:
# Median cluster-centric offset of the most massive ROCKSTAR subhalo found
# inside 10^-2 R_200, against the self-interaction cross section. One point per
# simulation; each cluster that hosts an inner object contributes its single
# most massive one, and N is the number of such clusters.
#
# The velocity-dependent run has no single sigma/m, so each of its objects gets
# its own effective cross section (see below) and the run enters as one point
# with bootstrapped error bars in *both* directions. The four CDM runs all sit
# at sigma/m = 0 and are dodged sideways so their error bars stay readable; the
# grey band marks the span they were dodged across.

SHOW_VDSIDM  = False      # include the velocity-dependent SIDM point
R_INNER      = 1e-2      # inner radius cut, in units of R_200
N_BOOT_OFF   = 5000      # bootstrap resamplings for the median CIs
SEED_OFF     = 0
CDM_DODGE    = np.linspace(-0.025, 0.025, 4)   # x offsets for the four CDM runs
CDM_BAND     = (-0.05, 0.05)                 # grey 'all CDM' band, in sigma/m

# Velocity-dependent SIDM cross section, Robertson et al. (2019) section 2.2:
# dsigma/dOmega = sigma_0 / (4 pi [1 + (v/w)^2 sin^2(theta/2)]^2). Integrating
# that against (1 - cos theta) gives the momentum-transfer cross section used
# here, which tends to sigma_0 below w and falls off above it.
SIGMA0_VD    = 3.04            # cm^2 g^-1
W_VD         = 560.0           # km s^-1
G_KPC        = 4.30117902e-6   # kpc (km/s)^2 / Msun  (rockstar's own Gc)

# (model, sigma/m, colour, marker, legend label)
OFFSET_SIMS = [
    ("CDMb",     0.0, 'red',      'o', r"BAHAMAS CDM"),
    ("TNG300-1", 0.0, 'purple',   '^', r"TNG300-1 CDM"),
    ("TNG300-2", 0.0, 'deeppink', '^', r"TNG300-2 CDM"),
    ("TNG300-3", 0.0, 'olive',    '^', r"TNG300-3 CDM"),
    ("SIDM0.1b", 0.1, 'green',    'o', r"BAHAMAS SIDM 0.1"),
    ("SIDM0.3b", 0.3, 'orange',   'o', r"BAHAMAS SIDM 0.3"),
    ("SIDM1b",   1.0, 'black',    'o', r"BAHAMAS SIDM 1.0"),
]


def inner_objects(model, r_inner=R_INNER):
    """Most massive subhalo inside r_inner, one row per cluster that has one.

    A handful of ROCKSTAR rows carry mgrav_bound == 0; they are dropped, since
    idxmax would otherwise return one of them for a cluster whose only inner
    object is such a row.
    """
    df = subhalos_df_by_model[model]
    r = df['r_R200'].to_numpy()
    inner = df[np.isfinite(r) & (r < r_inner) & (df['mgrav_bound'] > 0)]
    # idxmax picks the most massive object in each cluster that has one at all,
    # so clusters with an empty core simply drop out of the sample.
    return inner.loc[inner.groupby('cluster_id')['mgrav_bound'].idxmax()]


def inner_most_massive(model, r_inner=R_INNER):
    """r/R_200 of the most massive subhalo inside r_inner, one value per cluster."""
    return inner_objects(model, r_inner)['r_R200'].to_numpy()


def sigma_T_vd(v):
    """Momentum-transfer cross section [cm^2 g^-1] at relative velocity v [km/s]."""
    R = (np.asarray(v, dtype=float) / W_VD)**2
    return SIGMA0_VD * (2.0 / R**2) * (np.log1p(R) - R / (1.0 + R))


def vdsidm_sigma_and_offset(r_inner=R_INNER):
    """Per-object (sigma_T/m, r/R_200) for the vdSIDM inner objects.

    The scattering velocity is approximated by the circular velocity of the
    object, v = sqrt(G * mgrav_bound / rvmax). No h correction is applied on
    purpose: mgrav_bound is Msun/h and rvmax is comoving kpc/h, so h cancels in
    the ratio. That is rockstar's own convention -- universal_constants.h reads
    VMAX_CONST = sqrt(G*(Msun/h)/(Mpc/h)) -- and at z = 0 comoving = physical.
    """
    best = inner_objects('vdSIDMb', r_inner)
    best = best[best['rvmax'] > 0]
    v_rel = np.sqrt(G_KPC * best['mgrav_bound'].to_numpy() / best['rvmax'].to_numpy())
    return sigma_T_vd(v_rel), best['r_R200'].to_numpy()


def median_ci(x, n_boot=N_BOOT_OFF, seed=SEED_OFF):
    """Median plus its 68% and 95% bootstrap confidence intervals."""
    rng = np.random.default_rng(seed)
    boot = np.median(rng.choice(x, size=(n_boot, x.size), replace=True), axis=1)
    lo68, hi68 = np.percentile(boot, [16, 84])
    lo95, hi95 = np.percentile(boot, [2.5, 97.5])
    return np.median(x), lo68, hi68, lo95, hi95


def median_ci_paired(x, y, n_boot=N_BOOT_OFF, seed=SEED_OFF):
    """median_ci for x and y at once, resampling the same objects for both.

    The vdSIDM point has an uncertainty in both directions, and its cross
    section and offset are two properties of one object, so the replicates have
    to be drawn with a single set of indices rather than independently.
    """
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    out = []
    for v, boot in ((x, np.median(x[idx], axis=1)), (y, np.median(y[idx], axis=1))):
        lo68, hi68 = np.percentile(boot, [16, 84])
        lo95, hi95 = np.percentile(boot, [2.5, 97.5])
        out.append((np.median(v), lo68, hi68, lo95, hi95))
    return out


fig, ax = plt.subplots(figsize=(10, 7))

# Shade the range the sigma/m = 0 runs are dodged across.
ax.axvspan(*CDM_BAND, color='0.92', zorder=0)
ax.text(0.5 * (CDM_BAND[0] + CDM_BAND[1]), 0.955, 'all CDM\n' + r'($\sigma/m = 0$)',
        transform=ax.get_xaxis_transform(), ha='center', va='top',
        fontsize=13, color='0.4')

_n_cdm = 0
for model, xsec, color, marker, label in OFFSET_SIMS:
    r_inner = inner_most_massive(model)
    med, lo68, hi68, lo95, hi95 = median_ci(r_inner)

    if xsec == 0.0:
        x = CDM_DODGE[_n_cdm]
        _n_cdm += 1
    else:
        x = xsec

    # 95% interval behind, thin and grey; 68% in front, thick and coloured.
    ax.errorbar(x, med, yerr=[[med - lo95], [hi95 - med]],
                color='0.6', lw=1.3, capsize=0, zorder=2)
    ax.errorbar(x, med, yerr=[[med - lo68], [hi68 - med]],
                color=color, lw=3.5, capsize=5, capthick=2.0, zorder=3)
    ax.plot(x, med, marker=marker, ms=15, color=color,
            markeredgecolor='k', markeredgewidth=1.0, ls='none', zorder=4,
            label=f"{label}  ($N = {r_inner.size}$)")

    print(f"{model:10s} sigma/m={xsec:.1f}  N={r_inner.size:3d}  "
          f"median={med:.5f}  68%=[{lo68:.5f}, {hi68:.5f}]  95%=[{lo95:.5f}, {hi95:.5f}]")

x_upper = 1.09
if SHOW_VDSIDM:
    sig_vd, off_vd = vdsidm_sigma_and_offset()
    (xm, xlo68, xhi68, xlo95, xhi95), (ym, ylo68, yhi68, ylo95, yhi95) = \
        median_ci_paired(sig_vd, off_vd)

    ax.errorbar(xm, ym,
                xerr=[[xm - xlo95], [xhi95 - xm]], yerr=[[ym - ylo95], [yhi95 - ym]],
                color='0.6', lw=1.3, capsize=0, zorder=2)
    ax.errorbar(xm, ym,
                xerr=[[xm - xlo68], [xhi68 - xm]], yerr=[[ym - ylo68], [yhi68 - ym]],
                color='blue', lw=3.5, capsize=5, capthick=2.0, zorder=3)
    ax.plot(xm, ym, marker='o', ms=15, color='blue',
            markeredgecolor='k', markeredgewidth=1.0, ls='none', zorder=4,
            label=f"BAHAMAS SIDM Velocity-Dependent  ($N = {sig_vd.size}$)")

    x_upper = max(x_upper, xhi95 * 1.06)
    print(f"{'vdSIDMb':10s} sigma/m=eff  N={sig_vd.size:3d}  "
          f"median={ym:.5f}  68%=[{ylo68:.5f}, {yhi68:.5f}]  95%=[{ylo95:.5f}, {yhi95:.5f}]\n"
          f"{'':10s} sigma_T/m median={xm:.3f}  68%=[{xlo68:.3f}, {xhi68:.3f}]  "
          f"95%=[{xlo95:.3f}, {xhi95:.3f}]  per-object range=[{sig_vd.min():.3f}, {sig_vd.max():.3f}]")

f_size = 25

ax.set_xlabel(r'$\sigma/m$  [cm$^2$ g$^{-1}$]', fontsize=f_size)
ax.set_ylabel(r'$\langle r_{\rm sub}^{\rm rockstar}/R_{200} \rangle$', fontsize=f_size)

ax.set_xticks([0.0, 0.1, 0.3, 0.5, 1.0] + ([1.5, 2.0, 2.5] if SHOW_VDSIDM else []))
ax.set_xlim(-0.075, x_upper)
ax.set_ylim(0, None)
ax.tick_params(labelsize=14)

# Same inner-region criterion that labels the rightmost panel of Figure 4,
# sitting just above the legend.
#ax.text(0.92, 0.47, r'$r_{\rm sub}^{\rm rockstar}/R_{200} < 10^{-2}$', fontweight='semibold',
 #       transform=ax.transAxes, ha='right', va='bottom', fontsize=f_size)

ax.legend(loc='lower right', fontsize=16, frameon=False, numpoints=1)

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/offset_vs_cross_section.png',
    bbox_inches='tight',
    dpi=500,
    )
plt.show()


# Figure 6: Completeness, Purity, ARI versus $M_{200}$

### completeness / purity / ARI vs $M_{200}$

In [ ]:
DS_CASE_KEY = 'no_cuts'

def _binned_arrays(x, y, xbins, min_count=5):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    out = []
    centers = 0.5 * (xbins[:-1] + xbins[1:])

    for i, (lo, hi) in enumerate(zip(xbins[:-1], xbins[1:])):
        if i == len(xbins) - 2:
            m = np.isfinite(x) & np.isfinite(y) & (x >= lo) & (x <= hi)
        else:
            m = np.isfinite(x) & np.isfinite(y) & (x >= lo) & (x < hi)

        yy = y[m]
        if yy.size >= min_count:
            out.append(yy)
        else:
            out.append(np.array([]))

    return centers, out

def _plot_plain_binned_violin(ax, x, y, xbins, *, ylabel, ylim=None):
    centers, data = _binned_arrays(x, y, xbins)

    widths = 0.75 * np.diff(xbins)
    widths = np.full_like(centers, np.nanmedian(widths), dtype=float)

    valid = [d for d in data if len(d) > 0]
    valid_pos = [c for c, d in zip(centers, data) if len(d) > 0]

    # Scale each violin's width by its total count relative to the most-
    # populated bin, so the visual "amount" reflects how many points it holds
    # (matplotlib otherwise normalizes every violin to the same width).
    valid_counts = np.array([len(d) for d in valid], dtype=float)
    base_width = np.nanmedian(widths)
    valid_widths = list(base_width * valid_counts / valid_counts.max())

    parts = ax.violinplot(
        valid,
        positions=valid_pos,
        widths=valid_widths,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for body in parts['bodies']:
        body.set_facecolor('0.75')
        body.set_edgecolor('black')
        body.set_alpha(0.8)
        body.set_linewidth(0.8)

    # Connect the per-bin medians with a dashed red line and shade +/- 1 std.
    med_color = '#c1121f'
    medians = np.array([np.nanmedian(d) for d in valid], dtype=float)
    stds = np.array([np.nanstd(d) for d in valid], dtype=float)
    pos = np.asarray(valid_pos, dtype=float)

    pos = np.concatenate(([xbins[0]], pos, [xbins[-1]]))
    medians = np.concatenate((medians[:1], medians, medians[-1:]))
    stds = np.concatenate((stds[:1], stds, stds[-1:]))

    ax.fill_between(
        pos,
        medians - stds,
        medians + stds,
        color=med_color,
        alpha=0.2,
        linewidth=0,
        zorder=2,
    )
    ax.plot(
        pos,
        medians,
        linestyle='--',
        color=med_color,
        linewidth=3.5,
        zorder=3,
    )

    # Optional: show individual bin centers as x ticks
    ax.set_xticks(centers)
    ax.set_xticklabels([f'{c:.2f}' for c in centers])

    for b in xbins:
        ax.axvline(b, color='0.9', lw=0.8, zorder=0)

    f_size = 20
    ax.set_xlabel(r'$\log_{10}(M_{200}/M_\odot)$', fontsize=f_size)
    ax.set_ylabel(ylabel, fontsize=f_size)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.minorticks_on()


# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

ds_df = pd.read_csv(
    '/projects/mccleary_group/habjan.e/TNG/Data/data_DS+_stats/dsp_cases_stats.csv',
    comment='#',
)

ds_m = ds_df['m200'].to_numpy()
bins_m = np.linspace(14.25, 15.25, 6)

ys = [
    ('completeness', 'Completeness', (0.0, 1.0)),
    ('purity',       'Purity',       (0.0, 1.0)),
    ('ari',          'ARI',          (-0.3, 1.0)),
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(14, 4.5),
    sharex=True,
    constrained_layout=True
)

for ax, (ystem, ylabel, ylim) in zip(axes, ys):
    yvals = ds_df[f'{ystem}_{DS_CASE_KEY}'].to_numpy()

    _plot_plain_binned_violin(
        ax,
        ds_m,
        yvals,
        bins_m,
        ylabel=ylabel,
        ylim=ylim,
    )

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/c_p_ari_panels.png',
    bbox_inches='tight',
    dpi=500,
)

plt.show()

# Figure 7: Fragmentation fraction versus ROCKSTAR substructure radius

In [ ]:
# Fragmentation plain violin vs rockstar substructure radius.
# Uses only the min3_maxsqrtN substructure-size cut policy.

frag_df = pd.read_csv(
    '/projects/mccleary_group/habjan.e/TNG/Data/data_DS+_stats/fragmentation_stats.csv',
    comment='#',
)

# r/R_200 comes from the shared ROCKSTAR radii computed above (cMpc/h throughout,
# so no stray factor of 1/h), matched on (cluster, rockstar subhalo).
_rockstar_r = df_TNG300_1[['cluster_id', 'id', 'r_R200']].rename(
    columns={'id': 'rockstar_subhalo_id'}
).copy()
_rockstar_r['cluster_id'] = _rockstar_r['cluster_id'].astype(int)

frag_df = frag_df.merge(
    _rockstar_r,
    on=['cluster_id', 'rockstar_subhalo_id'],
    how='left',
)

frag_rR = frag_df['r_R200'].to_numpy()
frag_y = frag_df[f'fragmentation_{DS_CASE_KEY}'].to_numpy()

print(f"matched {np.isfinite(frag_rR).sum()}/{len(frag_rR)} rows; "
      f"r/R200 range {np.nanmin(frag_rR):.2f}-{np.nanmax(frag_rR):.2f}")

bins_rR = np.linspace(0, 3.5, 6)

fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)

_plot_plain_binned_violin(
    ax,
    frag_rR,
    frag_y,
    bins_rR,
    ylabel=r'$f_{\, a}^{\, \, frag}$',
    ylim=(0.0, 1.0),
)

f_size = 20
ax.set_xlabel(r'$r_{\rm sub}^{\rm rockstar} / R_{200}$', fontsize=f_size)
ax.set_ylabel(r'$f_{\, a}^{\, \, frag}$', fontsize=f_size)

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/fragmentation.png',
    bbox_inches='tight',
    dpi=500,
)

plt.show()


# Figure 8: $N_a$ versus $r_{sub} / R_{200}$

In [ ]:
# Figure 6: galaxy multiplicity N_a of each ROCKSTAR subhalo against its
# cluster-centric radius, coloured by how often DS+ misses the object entirely.
#
# N_a is the number of M_r < -18 subfind galaxies matched to a ROCKSTAR subhalo.
# It is the quantity that controls f^frag: f_a = 1 - best/N_a, so f_a = 1 means
# best = 0, i.e. DS+ placed every one of the N_a member galaxies in the
# background and recovered no part of the object in that projection.
#
# The matching is the one produced by rockstar/tng_subhalo_matching.py (mode of
# the ROCKSTAR membership of each subfind subhalo's first 50 DM particle IDs),
# and the magnitude cut is the same one TNG_DA.get_cluster_props applies.

_TNG_FIELD_DIR = '/projects/mccleary_group/habjan.e/TNG/Data/TNG_data/'
_MAG_CUT = -18.0

_grnr  = _h5_array(_TNG_FIELD_DIR + 'TNG300-1_SubhaloGrNr.hdf5',
                   'Subhalo/SubhaloGrNr')
_photo = _h5_array(_TNG_FIELD_DIR + 'TNG300-1_SubhaloStellarPhotometrics.hdf5',
                   'Subhalo/SubhaloStellarPhotometrics')

_na_rows = []
for _cid in range(100):
    _sub_ind = np.where(_grnr == _cid)[0]
    _bright  = _photo[_sub_ind, 4] < _MAG_CUT
    _matched = np.load(
        f'{_ROCKSTAR_BASE}/TNG300-1_rockstar_output/'
        f'matched_subhalo_members_{_cid}.npy'
    )
    # Unmatched galaxies come back as NaN and are simply not counted.
    _groups = _matched[_bright]
    _groups = _groups[~np.isnan(_groups)].astype(int)
    _ids, _cnt = np.unique(_groups, return_counts=True)
    _na_rows.append(pd.DataFrame({'cluster_id': _cid,
                                  'rockstar_subhalo_id': _ids,
                                  'N_a': _cnt}))

Na_df = pd.concat(_na_rows, ignore_index=True)
print(f"N_a computed for {len(Na_df)} (cluster, ROCKSTAR subhalo) pairs; "
      f"N_a = 1 for {(Na_df['N_a'] == 1).sum()} of them")

# ---------------------------------------------------------------------------
# Per-subhalo miss rate P_a: the fraction of the 1000 projections in which DS+
# recovered none of that subhalo's galaxies. r_R200 is already on frag_df from
# the Figure 5 cell.
# ---------------------------------------------------------------------------
_obj = (
    frag_df
    .groupby(['cluster_id', 'rockstar_subhalo_id'])
    .agg(P_miss=(f'fragmentation_{DS_CASE_KEY}', lambda s: float((s == 1).mean())),
         r_R200=('r_R200', 'first'))
    .reset_index()
    .merge(Na_df, on=['cluster_id', 'rockstar_subhalo_id'], how='left')
)
print(f"{len(_obj)} subhalos; N_a missing for {_obj['N_a'].isna().sum()}")

# ---------------------------------------------------------------------------
# Binning. Rows are integer N_a, with everything above NA_TOP folded into a
# single top row so no subhalo is left off the panel. The innermost column
# accumulates r/R_200 <= R_IN: below that radius sits the central/BCG-scale
# object, which is a different class of thing and would otherwise smear over
# three decades of empty log space. Columns above R_IN step by 0.25 dex.
# ---------------------------------------------------------------------------
NA_TOP = 10
R_IN   = 0.1
DEX    = 0.25

_ybin   = np.minimum(_obj['N_a'].to_numpy(), NA_TOP)
_yedges = np.arange(0.5, NA_TOP + 1.0, 1.0)
_xhi    = 10 ** np.arange(np.log10(R_IN), 0.7501, DEX)
# The accumulation cell is drawn one bin-width wide so every cell is the same
# size; its left edge is labelled 0 because it really does hold [0, R_IN].
_xedges = np.concatenate(([10 ** (np.log10(R_IN) - DEX)], _xhi))

_r  = _obj['r_R200'].to_numpy()
_P  = _obj['P_miss'].to_numpy()

_grid  = np.full((len(_yedges) - 1, len(_xedges) - 1), np.nan)
_count = np.zeros_like(_grid)
for _i in range(len(_yedges) - 1):
    _in_row = _ybin == _i + 1
    for _j in range(len(_xedges) - 1):
        if _j == 0:
            _in_col = _r <= R_IN
        else:
            _in_col = (_r > _xedges[_j]) & (_r <= _xedges[_j + 1])
        _sel = _in_row & _in_col & np.isfinite(_P)
        _count[_i, _j] = _sel.sum()
        if _sel.sum() > 0:                      # empty cells stay blank
            _grid[_i, _j] = np.median(_P[_sel])

print(f"{int(np.isfinite(_grid).sum())} of {_grid.size} cells occupied")

f_size = 25

fig, ax = plt.subplots(figsize=(8, 5.6), constrained_layout=True)

_cmap = LinearSegmentedColormap.from_list(
    'gray_red', ['#c1121f', '0.75']  # dark gray -> muted brick red
)
_cmap.set_bad('white')
_mesh = ax.pcolormesh(_xedges, _yedges, np.ma.masked_invalid(_grid),
                      cmap=_cmap, vmin=0.0, vmax=1.0,
                      edgecolors='0.88', linewidth=0.45, shading='flat')

_cb = fig.colorbar(_mesh, ax=ax, pad=0.015)
_cb.set_label(r'median $P_a\left(f_{\, a}^{\, \, frag} = 1\right)$', fontsize=20)
_cb.ax.tick_params(labelsize=11)

ax.set_xscale('log')
ax.set_xlim(_xedges[0], _xedges[-1])
ax.set_ylim(_yedges[0], _yedges[-1])

ax.set_yticks(np.arange(1, NA_TOP + 1))
ax.set_yticklabels([str(_v) for _v in range(1, NA_TOP)] + [rf'$\geq\!{NA_TOP}$'],
                   rotation=0, fontsize=12)

# Every x tick sits on a cell edge, written as a power of ten.
ax.set_xticks([_xedges[0]] + list(_xhi))
ax.set_xticks([], minor=True)
_exps = np.log10(_xhi)
ax.set_xticklabels(
    ['$0$'] + [rf'$10^{{{round(_e)}}}$' if abs(_e - round(_e)) < 1e-9
               else rf'$10^{{{_e:.2f}}}$' for _e in _exps],
    fontsize=12,
)

ax.set_xlabel(r'$r_{\rm sub}^{\rm rockstar} / R_{200}$', fontsize=f_size)
ax.set_ylabel(r'$N_a$', fontsize=f_size)
ax.tick_params(axis='y', labelsize=13)

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/Na_vs_radius.png',
    bbox_inches='tight',
    dpi=500,
)

plt.show()


# Figure 9: Merging fraction vs DS+ substructure radius

In [ ]:
# Merging plain violin vs r_sub^{DS+} / R_200.
# Uses only the min3_maxsqrtN substructure-size cut policy.

merge_df = pd.read_csv(
    '/projects/mccleary_group/habjan.e/TNG/Data/data_DS+_stats/merging_stats.csv',
    comment='#',
)

merge_r_Mpc = merge_df['substructure_com_r'].to_numpy() / 1e3   # kpc -> Mpc
merge_rR = merge_r_Mpc / _R200_TNG_cMpc[merge_df['cluster_id'].to_numpy()]
merge_y = merge_df[f'merging_{DS_CASE_KEY}'].to_numpy()

bins_rR = np.linspace(0, 2.5, 6)

fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)

_plot_plain_binned_violin(
    ax,
    merge_rR,
    merge_y,
    bins_rR,
    ylabel=r'$f_{\, b}^{\, \, merge}$',
    ylim=(0.0, 1.0),
)

f_size = 20
ax.set_xlabel(r'$r_{\rm sub}^{\rm DS+} / R_{200}$', fontsize=f_size)
ax.set_ylabel(r'$f_{\, b}^{\, \, merge}$', fontsize=f_size)

fig.savefig(
    '/home/habjan.e/TNG/TNG_cluster_dynamics/sub_figures/merging.png',
    bbox_inches='tight',
    dpi=500,
)

plt.show()